In [1]:
from pathlib import Path
import json
import re
import pandas as pd

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

NORMALIZED_DOCS_PATH = AI_LAB_ROOT / "normalized" / "docs.jsonl"
REPORTS_DIR = AI_LAB_ROOT / "reports"
DATASETS_DIR = AI_LAB_ROOT / "datasets"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_CSV_PATH = REPORTS_DIR / "kb_candidate_blocks_v1.csv"
REVIEW_QUEUE_PATH = REPORTS_DIR / "kb_review_queue_v1.csv"
MEDICAL_KB_PATH = DATASETS_DIR / "medical_kb_v1.json"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("NORMALIZED_DOCS_PATH =", NORMALIZED_DOCS_PATH)

AI_LAB_ROOT = D:\homelab\ai_lab
NORMALIZED_DOCS_PATH = D:\homelab\ai_lab\normalized\docs.jsonl


In [2]:
def read_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

docs = read_jsonl(NORMALIZED_DOCS_PATH)
df_docs = pd.DataFrame(docs)

print("Tổng số docs:", len(df_docs))
df_docs[["source_id", "title", "section", "risk_level", "use_in_v1", "review_required"]]

Tổng số docs: 8


,source_id,title,section,risk_level,use_in_v1,review_required
0,blood_tests,Blood tests\n - NHS,test_explainers,low,True,False
1,chest_pain,Chest pain\n - NHS,red_flags,high,True,True
2,shortness_of_breath,Shortness of breath\n - NHS,red_flags,high,True,True
3,cdc_dpdx_blood_collection,CDC - DPDx - Diagnostic Procedures - Blood Spe...,sample_collection,medium,False,True
4,cdc_specimen_packing_and_shipping,source,lab_ops,low,False,True
5,nice_sepsis_overview,source,red_flags,high,True,True
6,nice_sepsis_guideline,Overview | Suspected sepsis in people aged 16 ...,red_flags,high,True,True
7,who_infectious_shipping_guidance,source,lab_ops,low,False,True


In [3]:
V1_SECTIONS = {"test_explainers", "pre_test_guides", "red_flags"}

docs_v1 = [
    d for d in docs
    if d.get("use_in_v1", False) and d.get("section") in V1_SECTIONS
]

df_v1 = pd.DataFrame(docs_v1)
print("Số docs dùng cho health_rag v1:", len(df_v1))
df_v1[["source_id", "title", "section", "risk_level"]]

Số docs dùng cho health_rag v1: 5


,source_id,title,section,risk_level
0,blood_tests,Blood tests\n - NHS,test_explainers,low
1,chest_pain,Chest pain\n - NHS,red_flags,high
2,shortness_of_breath,Shortness of breath\n - NHS,red_flags,high
3,nice_sepsis_overview,source,red_flags,high
4,nice_sepsis_guideline,Overview | Suspected sepsis in people aged 16 ...,red_flags,high


In [4]:
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_into_candidate_blocks(text: str, min_len=120, max_len=1400):
    text = clean_text(text)

    raw_blocks = re.split(r"\n\s*\n", text)
    raw_blocks = [b.strip() for b in raw_blocks if b.strip()]

    candidates = []
    buffer = ""

    for block in raw_blocks:
        # bỏ block quá ngắn kiểu heading rời rạc
        if len(block) < 40:
            continue

        if not buffer:
            buffer = block
        elif len(buffer) + 2 + len(block) <= max_len:
            buffer += "\n\n" + block
        else:
            if len(buffer) >= min_len:
                candidates.append(buffer.strip())
            buffer = block

    if buffer and len(buffer) >= min_len:
        candidates.append(buffer.strip())

    return candidates

In [5]:
candidate_rows = []

for doc in docs_v1:
    blocks = split_into_candidate_blocks(doc["content"])

    for idx, block in enumerate(blocks, start=1):
        candidate_rows.append({
            "candidate_id": f"{doc['source_id']}_cand_{idx:03d}",
            "doc_id": doc["doc_id"],
            "source_id": doc["source_id"],
            "source_name": doc["source_name"],
            "source_url": doc["source_url"],
            "section": doc["section"],
            "risk_level": doc["risk_level"],
            "review_required": doc["review_required"],
            "block_index": idx,
            "char_count": len(block),
            "raw_block": block,

            # cột để bạn sửa tay
            "keep": 0,
            "final_title": "",
            "final_content": "",
            "tags": "",
            "keywords": "",
            "test_types": "",
            "faq_type": "",
            "safety_notes": "",
            "locale": "vi-VN",
            "language": "vi",
            "review_status": "pending"
        })

df_candidates = pd.DataFrame(candidate_rows)
df_candidates.to_csv(CANDIDATE_CSV_PATH, index=False, encoding="utf-8-sig")

print("Đã ghi:", CANDIDATE_CSV_PATH)
print("Số candidate blocks:", len(df_candidates))
df_candidates.head(10)

Đã ghi: D:\homelab\ai_lab\reports\kb_candidate_blocks_v1.csv
Số candidate blocks: 80


,candidate_id,doc_id,source_id,source_name,source_url,section,risk_level,review_required,block_index,char_count,...,final_title,final_content,tags,keywords,test_types,faq_type,safety_notes,locale,language,review_status
0,blood_tests_cand_001,blood_tests_doc_001,blood_tests,NHS,https://www.nhs.uk/tests-and-treatments/blood-...,test_explainers,low,False,1,4097,...,,,,,,,,vi-VN,vi,pending
1,chest_pain_cand_001,chest_pain_doc_001,chest_pain,NHS,https://www.nhs.uk/symptoms/chest-pain/,red_flags,high,True,1,2459,...,,,,,,,,vi-VN,vi,pending
2,shortness_of_breath_cand_001,shortness_of_breath_doc_001,shortness_of_breath,NHS,https://www.nhs.uk/symptoms/shortness-of-breath/,red_flags,high,True,1,3006,...,,,,,,,,vi-VN,vi,pending
3,nice_sepsis_overview_cand_001,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,1,297,...,,,,,,,,vi-VN,vi,pending
4,nice_sepsis_overview_cand_002,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,2,1892,...,,,,,,,,vi-VN,vi,pending
5,nice_sepsis_overview_cand_003,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,3,3852,...,,,,,,,,vi-VN,vi,pending
6,nice_sepsis_overview_cand_004,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,4,3343,...,,,,,,,,vi-VN,vi,pending
7,nice_sepsis_overview_cand_005,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,5,1180,...,,,,,,,,vi-VN,vi,pending
8,nice_sepsis_overview_cand_006,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,6,1386,...,,,,,,,,vi-VN,vi,pending
9,nice_sepsis_overview_cand_007,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,7,1809,...,,,,,,,,vi-VN,vi,pending


In [6]:
for _, row in df_candidates.head(8).iterrows():
    print("=" * 100)
    print("SOURCE:", row["source_id"])
    print("SECTION:", row["section"])
    print("CANDIDATE:", row["candidate_id"])
    print("RAW BLOCK:\n")
    print(row["raw_block"][:2000])
    print("\n")

SOURCE: blood_tests
SECTION: test_explainers
CANDIDATE: blood_tests_cand_001
RAW BLOCK:

A blood test is often done to check your health, or to find out why you're having certain symptoms. It involves having a small amount of your blood taken for testing.
Why a blood test is done
There are lots of reasons why you may need a blood test.
A blood test may be done to:
check your general health
find out if symptoms you're having are caused by certain conditions
find out if you're more likely to get a condition
find out how well a condition is being treated or managed
How to book a blood test
If a healthcare professional such as a GP, nurse or specialist thinks you need a blood test they will tell you how to book one.
Some GP surgeries or hospitals allow you to book a blood test online.
Types of blood test
Blood tests can check for different things depending on your symptoms, any conditions you have, and any medicines you’re taking.
Common types of blood test
Test
Why it's done
Full blood co

In [14]:
# đọc lại csv sau khi sửa
edited_df = pd.read_csv(CANDIDATE_CSV_PATH)

print("Tổng số rows:", len(edited_df))
edited_df.head()

Tổng số rows: 91


,candidate_id,doc_id,source_id,source_name,source_url,section,risk_level,review_required,block_index,char_count,...,final_title,final_content,tags,keywords,test_types,faq_type,safety_notes,locale,language,review_status
0,blood_tests_cand_001,blood_tests_doc_001,blood_tests,NHS,https://www.nhs.uk/tests-and-treatments/blood-...,test_explainers,low,False,1,4097,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vi-VN,vi,pending
1,chest_pain_cand_001,chest_pain_doc_001,chest_pain,NHS,https://www.nhs.uk/symptoms/chest-pain/,red_flags,high,True,1,2459,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vi-VN,vi,pending
2,shortness_of_breath_cand_001,shortness_of_breath_doc_001,shortness_of_breath,NHS,https://www.nhs.uk/symptoms/shortness-of-breath/,red_flags,high,True,1,3006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vi-VN,vi,pending
3,nice_sepsis_overview_cand_001,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,1,297,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vi-VN,vi,pending
4,nice_sepsis_overview_cand_002,nice_sepsis_overview_doc_001,nice_sepsis_overview,NICE,https://www.nice.org.uk/guidance/ng253,red_flags,high,True,2,1892,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vi-VN,vi,pending


In [15]:
import pandas as pd

edited_df = pd.read_csv(CANDIDATE_CSV_PATH, encoding="utf-8-sig")

# Kiểm tra cột bắt buộc
required_cols = ["keep", "final_title", "final_content", "source_id", "section", "risk_level", "review_status"]
for col in required_cols:
    if col not in edited_df.columns:
        raise ValueError(f"Thiếu cột bắt buộc: {col}")

# Chuẩn hóa keep về số
edited_df["keep"] = pd.to_numeric(edited_df["keep"], errors="coerce").fillna(0).astype(int)

# Chuẩn hóa các cột text về string
text_cols = ["final_title", "final_content", "review_status"]
for col in text_cols:
    edited_df[col] = edited_df[col].fillna("").astype(str).str.strip()

selected = edited_df[edited_df["keep"] == 1].copy()
print("Số block được chọn:", len(selected))

review_queue = selected[
    (selected["final_title"] == "") |
    (selected["final_content"] == "") |
    (selected["review_status"].str.lower() != "approved")
].copy()

review_queue.to_csv(REVIEW_QUEUE_PATH, index=False, encoding="utf-8-sig")
print("Đã ghi review queue:", REVIEW_QUEUE_PATH)
print("Số item chưa đủ điều kiện export:", len(review_queue))

Số block được chọn: 15
Đã ghi review queue: D:\homelab\ai_lab\reports\kb_review_queue_v1.csv
Số item chưa đủ điều kiện export: 0


In [16]:
approved = selected[
    selected["final_title"].fillna("").str.strip().ne("") &
    selected["final_content"].fillna("").str.strip().ne("") &
    selected["review_status"].fillna("").str.strip().eq("approved")
].copy()

print("Số item approved để export:", len(approved))
approved[["candidate_id", "final_title", "section", "risk_level"]].head()

Số item approved để export: 15


,candidate_id,final_title,section,risk_level
8,nice_sepsis_overview_cand_006,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,high
14,nice_sepsis_overview_cand_012,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,high
22,nice_sepsis_overview_cand_020,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,high
25,nice_sepsis_overview_cand_023,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,high
80,blood_tests_cand_001_a,Xét nghiệm máu là gì,test_explainers,low


In [17]:
def parse_csv_list(value):
    if pd.isna(value):
        return []
    parts = [x.strip() for x in str(value).split(",")]
    return [x for x in parts if x]

In [18]:
medical_kb = []

for i, row in enumerate(approved.itertuples(index=False), start=1):
    medical_kb.append({
        "id": f"kb_v1_{i:03d}",
        "doc_id": row.doc_id,
        "source_id": row.source_id,
        "source_name": row.source_name,
        "source_url": row.source_url,
        "section": row.section,
        "title": str(row.final_title).strip(),
        "content": str(row.final_content).strip(),
        "source_excerpt": str(row.raw_block).strip()[:1200],
        "language": row.language if pd.notna(row.language) else "vi",
        "locale": row.locale if pd.notna(row.locale) else "vi-VN",
        "risk_level": row.risk_level,
        "tags": parse_csv_list(row.tags),
        "keywords": parse_csv_list(row.keywords),
        "test_types": parse_csv_list(row.test_types),
        "faq_type": str(row.faq_type).strip() if pd.notna(row.faq_type) else "",
        "safety_notes": str(row.safety_notes).strip() if pd.notna(row.safety_notes) else "",
        "review_status": str(row.review_status).strip(),
        "use_in_v1": True
    })

with open(MEDICAL_KB_PATH, "w", encoding="utf-8") as f:
    json.dump(medical_kb, f, ensure_ascii=False, indent=2)

print("Đã ghi:", MEDICAL_KB_PATH)
print("Số knowledge items:", len(medical_kb))

Đã ghi: D:\homelab\ai_lab\datasets\medical_kb_v1.json
Số knowledge items: 15


In [19]:
pd.DataFrame(medical_kb)[["id", "source_id", "title", "section", "risk_level"]].head(20)

,id,source_id,title,section,risk_level
0,kb_v1_001,nice_sepsis_overview,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,high
1,kb_v1_002,nice_sepsis_overview,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,high
2,kb_v1_003,nice_sepsis_overview,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,high
3,kb_v1_004,nice_sepsis_overview,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,high
4,kb_v1_005,blood_tests,Xét nghiệm máu là gì,test_explainers,low
5,kb_v1_006,blood_tests,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,low
6,kb_v1_007,blood_tests,Một số loại xét nghiệm máu thường gặp,test_explainers,low
7,kb_v1_008,blood_tests,Cần chuẩn bị gì trước khi xét nghiệm máu,test_explainers,low
8,kb_v1_009,blood_tests,Sau khi xét nghiệm máu và khi nhận kết quả,test_explainers,low
9,kb_v1_010,chest_pain,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,high
